# Campus Placement Eligibility Predictor — Exploratory Data Analysis

This notebook explores the dataset used to train the placement eligibility model:
- Distribution of features (CGPA, skills, internships, projects)
- Class balance (eligible vs not eligible)
- Relationships between features and the target
- Correlations between all variables
- Feature engineering preview


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df = pd.read_csv("data.csv")
df.head()

## 1. Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe()

## 2. Target Class Balance

In [ ]:
counts = df["eligible"].value_counts().sort_index()
print(counts)

fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(x="eligible", hue="eligible", data=df, palette="viridis", legend=False, ax=ax)
ax.set_xticks([0,1])
ax.set_xticklabels(["Not Eligible (0)", "Eligible (1)"])
ax.set_title("Class Distribution")
plt.show()

## 3. Feature Distributions

In [ ]:
numeric_cols = ["cgpa", "skills", "internships", "projects"]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

## 4. CGPA vs Eligibility

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.boxplot(x="eligible", y="cgpa", hue="eligible", data=df, palette="Set2", legend=False, ax=ax)
ax.set_xticks([0,1])
ax.set_xticklabels(["Not Eligible (0)", "Eligible (1)"])
ax.set_title("CGPA vs Placement Eligibility")
plt.show()

print("Average CGPA (Eligible):", df[df.eligible==1]["cgpa"].mean().round(2))
print("Average CGPA (Not Eligible):", df[df.eligible==0]["cgpa"].mean().round(2))

## 5. Eligibility Rate by Skills / Internships / Projects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax, col in zip(axes, ["skills", "internships", "projects"]):
    rate = df.groupby(col)["eligible"].mean()
    sns.barplot(x=rate.index, y=rate.values, hue=rate.index, palette="crest", legend=False, ax=ax)
    ax.set_ylabel("Eligibility Rate")
    ax.set_title(f"Eligibility Rate by {col}")
    ax.set_ylim(0,1)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Heatmap")
plt.show()

## 7. Pairwise Relationships

In [ ]:
sns.pairplot(df, vars=numeric_cols, hue="eligible", palette="husl", diag_kind="kde")
plt.show()

## 8. Feature Engineering Preview

Three new features are created before training:

- **experience_score** = skills + internships + projects
- **cgpa_band** = categorical CGPA bucket (Low / Medium / High)
- **cgpa_experience_interaction** = cgpa * experience_score


In [ ]:
from features import add_engineered_features

df_fe = add_engineered_features(df)
df_fe.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.boxplot(x="eligible", y="experience_score", hue="eligible", data=df_fe,
            palette="Set3", legend=False, ax=ax)
ax.set_xticks([0,1])
ax.set_xticklabels(["Not Eligible (0)", "Eligible (1)"])
ax.set_title("Experience Score vs Eligibility")
plt.show()